# Parquet Explorer

Ham parquet dosyalarını incelemek için bir araç: schema, null oranları ve interaktif
tablo (sırala, ara, filtrele, sayfala, CSV indir). Tablolar `itables` ile çiziliyor.

**Bu bir araç, ölçüm değil.** `data-quality-dashboard.ipynb` dosyasından ayrı duruyor,
çünkü doğrulama notebook'unun içinde bir veri tarayıcısı taşımak onu ~2MB'lik bir dosyaya
çeviriyordu ve sunumda tablo kaydırmak anlatının dikkatini dağıtıyor.

**Üç sınır:**

- **Satır sınırı açık.** Tablo ilk `PREVIEW_ROWS` satırla sınırlı ve sınır caption'da
  yazıyor. itables'ın varsayılan davranışı, 64KB'ı aşan tabloyu sessizce örneklemek;
  bunu kapattık (`itables.options.maxBytes = 0`), çünkü sessiz küçültme "veriyi gördüm"
  yanlış izlenim veriyor.
- **Merged dosya koşu bitince yazılıyor.** `flows.parquet` ve `events.parquet`
  yokken bölüm en yeni `.partNNNN.parquet` dosyasına düşüyor ve bunu ekrana yazıyor.
- **itables JavaScript istiyor.** JS ve CSS tablo çıktısına base64 olarak gömülüyor
  (tarayıcıda internet gerekmiyor), ama JS çalışmazsa — güvensiz (untrusted) notebook,
  statik PDF — tablo gizli kalıyor.

In [ ]:
import sys
from pathlib import Path

# `style.py` bu notebook'un yanında (notebooks/). Kernel cwd'si normalde notebook'un
# dizinidir (JupyterLab ve nbconvert ikisi de böyle); JupyterLab'in kökü repo kökü
# verilmişse diye tek adım aşağısı da denenir.
HERE = Path.cwd().resolve()
if not (HERE / "style.py").is_file():
    HERE = HERE / "notebooks"
sys.path.insert(0, str(HERE))

import style

import pandas as pd
from IPython.display import display

import itables
import pyarrow.parquet as pq

REPO = style.bootstrap()      # repo kökü + src/ import yolunda


def stamp(path):
    """Dosyanın mtime'ı (UTC)."""
    return pd.Timestamp(path.stat().st_mtime, unit="s", tz="UTC")

# all_interactive=False: yalnızca açıkça `show()` denen tablolar interaktif olur.
itables.init_notebook_mode(all_interactive=False)

# Sessiz downsampling KAPALI — satır sınırını biz koyuyoruz ve caption'da yazıyoruz.
itables.options.maxBytes = 0

PREVIEW_ROWS = 1000


def parquet_rows(path):
    """Satır sayısını yalnızca footer'dan okur — part dosyalarını yüklemeye gerek yok."""
    try:
        return pq.ParquetFile(path).metadata.num_rows
    except Exception:
        return None


def inventory(name, merged, parts):
    """Bir veri setinin merged dosyası + part dosyaları envanteri."""
    rows = []
    if merged.is_file():
        rows.append({"source": merged.name, "kind": "merged", "rows": parquet_rows(merged),
                     "MB": round(merged.stat().st_size / 1e6, 2),
                     "mtime (UTC)": f"{stamp(merged):%Y-%m-%d %H:%M}"})
    if parts:
        rows.append({"source": f"{name}.parquet.part*.parquet", "kind": f"{len(parts)} part",
                     "rows": sum(parquet_rows(p) or 0 for p in parts),
                     "MB": round(sum(p.stat().st_size for p in parts) / 1e6, 2),
                     "mtime (UTC)": f"{stamp(max(parts, key=lambda p: p.stat().st_mtime)):%Y-%m-%d %H:%M}"})
    if not rows:
        rows.append({"source": name, "kind": "YOK", "rows": None, "MB": None, "mtime (UTC)": "—"})
    return rows


def preview(title, merged, parts):
    """Schema + interaktif tablo. Merged dosya yoksa (koşu sürüyorsa) en yeni part'ı gösterir."""
    if merged.is_file():
        path, total = merged, parquet_rows(merged)
    elif parts:
        path = max(parts, key=lambda p: p.stat().st_mtime)
        total = sum(parquet_rows(p) or 0 for p in parts)
        print(f"{merged.name} henüz yok (koşu sürüyor olabilir) — "
              f"en yeni part gösteriliyor: {path.name}")
    else:
        print(f"— {title}: veri yok")
        return

    d = pd.read_parquet(path)
    print(f"\n=== {title} ===  {path.name}  ·  {stamp(path):%Y-%m-%d %H:%M} UTC")
    print(f"bu dosya: {len(d)} satır × {len(d.columns)} sütun   |   kaynak toplam: {total:,}")

    schema = pd.DataFrame({"dtype": d.dtypes.astype(str),
                           "non-null": d.notna().sum(),
                           "null %": (d.isna().mean() * 100).round(1)})
    display(schema)
    itables.show(d.head(PREVIEW_ROWS),
                 caption=f"{title} — ilk {min(PREVIEW_ROWS, len(d))} satır "
                         f"(kaynak toplam {total:,})")


RAW_DIR = REPO / "src/data/raw"
FEATURES = REPO / "src/data/features/features.parquet"
FLOW_MERGED, EVENT_MERGED = RAW_DIR / "flows.parquet", RAW_DIR / "events.parquet"
FLOW_PARTS = sorted(RAW_DIR.glob("flows.parquet.part*.parquet"))
EVENT_PARTS = sorted(RAW_DIR.glob("events.parquet.part*.parquet"))

print(f"raw dizini: {RAW_DIR}")

In [ ]:
import pandas as pd

# Envanter: hangi veri var, ne kadar, ne zaman yazıldı.
rows = []
for name, merged, parts in [("flows", FLOW_MERGED, FLOW_PARTS),
                            ("events", EVENT_MERGED, EVENT_PARTS),
                            ("features", FEATURES, [])]:
    rows += [{"dataset": name, **r} for r in inventory(name, merged, parts)]
pd.DataFrame(rows)

In [ ]:
preview("flows — Hubble network akışları", FLOW_MERGED, FLOW_PARTS)

In [ ]:
preview("events — Tetragon runtime olayları", EVENT_MERGED, EVENT_PARTS)

In [ ]:
preview("features — ML-ready feature vektörleri", FEATURES, [])

## Not: şema ile `src/README.md` uyuşmuyor

Bu gezgin, `src/README.md` içindeki şema tablosunun veriyle uyuşmadığını ortaya çıkardı.
`flows` ve `events` gerçekte README'de yazandan daha fazla sütun taşıyor. Ayrıca
README'de belgelenen `l7_type` ve `l7_status` sütunları veride yok (FAZ 3.5'te L7 kapsam
dışı ilan edilmişti) ve `process_id` diye belgelenen sütunun adı veride `pid`.